In [30]:
import pandas as pd 
import json
import os
import time
from typing import Callable
import platform
#from hdbcli import dbapi
from fileinput import filename
import csv
from datetime import datetime
from datetime import date




#from gen_ai_hub.proxy import get_proxy_client
#from ai_core_sdk.ai_core_v2_client import AICoreV2Client
#from ai_api_client_sdk.models.status import Status
#from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.llm import LLM
#from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
#from gen_ai_hub.orchestration.models.template import Template, TemplateValue
#from gen_ai_hub.orchestration.models.config import OrchestrationConfig
#from gen_ai_hub.orchestration.models.data_masking import DataMasking
#from gen_ai_hub.orchestration.models.sap_data_privacy_integration import (
#    SAPDataPrivacyIntegration, MaskingMethod, ProfileEntity )
from gen_ai_hub.proxy.langchain.init_models import init_llm
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

vaddress="" 
port=443
vuser=""
vpassword=""

# STEP 1: Load AI Core credentials
def load_ai_core_credentials():
    with open('/Users/I871395/Downloads/VKExplore/AgenticAI/azure-evm-config.json', 'r') as f:
        svcKey = json.load(f)

    os.environ["AICORE_AUTH_URL"] = svcKey["AICORE_AUTH_URL"]
    os.environ["AICORE_CLIENT_ID"] = svcKey["AICORE_CLIENT_ID"]
    os.environ["AICORE_CLIENT_SECRET"] = svcKey["AICORE_CLIENT_SECRET"]
    os.environ["AICORE_RESOURCE_GROUP"] = "default"
    os.environ["AICORE_BASE_URL"] = svcKey["AICORE_BASE_URL"]
    os.environ["HANA_ADDRESS"] = svcKey["HANA_HOST"]
    os.environ["HANA_USER"] = svcKey["HANA_USER"]
    os.environ["HANA_PASSWORD"] = svcKey["HANA_PASSWORD_VDB"]
    vaddress=svcKey["HANA_HOST"]
    vuser=svcKey["HANA_USER"]
    vpassword=svcKey["HANA_PASSWORD_VDB"]

    return vuser,vpassword,vaddress


In [31]:
a = load_ai_core_credentials()
print(a)

('DBADMIN', 'SFappopsai2025!@#', '37460116-6bcc-412a-a29d-1797a74faf62.hna1.canary-eu10.hanacloud.ondemand.com')


In [33]:
# Function to chat with table data

llm = init_llm('gpt-4o-mini', max_tokens=4096, temperature=0.0)
print(llm)

profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<gen_ai_hub.proxy.native.openai.clients.ChatCompletions object at 0x115e61250> async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncChatCompletions object at 0x115e627b0> root_client=<gen_ai_hub.proxy.native.openai.clients.OpenAI object at 0x116007610> root_async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncOpenAI object at 0x1160074d0> model_name='gpt-4o-mini' temperature=0.0 model_kwa

In [34]:
print(llm)

profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<gen_ai_hub.proxy.native.openai.clients.ChatCompletions object at 0x115e61250> async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncChatCompletions object at 0x115e627b0> root_client=<gen_ai_hub.proxy.native.openai.clients.OpenAI object at 0x116007610> root_async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncOpenAI object at 0x1160074d0> model_name='gpt-4o-mini' temperature=0.0 model_kwa

In [39]:
print(llm.get_name())


ChatOpenAI


In [41]:
messages = [SystemMessage(content="You are a Expert"), HumanMessage("tell me current time in san francisco")]
print(llm.invoke(messages))


content="I can't provide real-time information, including the current time. However, San Francisco is in the Pacific Time Zone (PT). You can find the current time by checking a world clock or your device's clock set to that time zone. If you need help with time zone conversions or anything else, feel free to ask!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 23, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DZ90KUrdkz1GV2DgAjmDK4npBMRJl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dcd74-9931-72d3-a140-2d39186226cd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_token

In [10]:
from langchain_core.prompts import PromptTemplate
user_input = input("Enter the topic you want to get answered")
dynamic_input = PromptTemplate.from_template("Answer for the query on {topic}")
ready_prompt = dynamic_input.invoke({"topic", user_input})
llm.invoke(ready_prompt).content

'The term "model context protocol in AI" is not a widely recognized or standard term in the field of artificial intelligence as of my last knowledge update in October 2023. However, I can break down the components of the phrase to provide some insights that may be relevant.\n\n1. **Model**: In AI, a model typically refers to a mathematical representation or algorithm that is trained on data to perform specific tasks, such as classification, regression, or generation.\n\n2. **Context**: Context in AI can refer to the surrounding information or circumstances that influence the interpretation of data or the behavior of a model. For example, in natural language processing (NLP), the context of a word can significantly affect its meaning.\n\n3. **Protocol**: A protocol generally refers to a set of rules or procedures for communication or data exchange. In AI, protocols can be related to how models interact with each other, how data is shared, or how systems are integrated.\n\nPutting these 

In [8]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages(['system' ,'You are Expert Programmer' , 'user' ,'tell me labout {topic}'])
ready_prompt = chat_template.invoke({"topic" : "AI"})
ready_prompt.messages

[HumanMessage(content='system', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='You are Expert Programmer', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='user', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='tell me labout AI', additional_kwargs={}, response_metadata={})]

In [9]:
from pydantic import BaseModel
class llm_schema(BaseModel):
    setup : str
    punchline : str

obj = llm_schema(**{'setup': 'aa', 'punchline' : 'some punchline'})
print(obj)

setup='aa' punchline='some punchline'


In [10]:
from pydantic import BaseModel , Field
class llm_schema(BaseModel):
    setup : str = Field(description="aa")
    punchline : str = Field(description="aa")

obj = llm_schema(**{'setup': 'some setup', 'punchline' : 'some punchline'})
llmstructuredop = llm.with_structured_output(llm_schema)
result = llmstructuredop.invoke("What is capital of india")
print(result.setup)
print(result.punchline)

Question: What is the capital of India?
New Delhi.
